In [46]:
import pandas as pd
import numpy as np
import pandas_ta as ta
from datetime import time
from quant_backtester_core import QuantEngineV2, StrategyConfig
from quant_reporting import QuantReporter

# --- 1. CARGA Y LIMPIEZA ---
# (Asegúrate de que la ruta sea la correcta en tu nueva instancia)
FILE_PATH_PARQUET = '/home/quant/data/processed/gc_1m_raw_continuous.parquet'
df = pd.read_parquet(FILE_PATH_PARQUET)

df['Timestamp_NY'] = pd.to_datetime(df['Timestamp_NY'])
df = df.sort_values('Timestamp_NY').reset_index(drop=True)

# --- 2. INDICADORES (Lógica Pine Script) ---
df['ema11'] = ta.ema(df['Close_adj'], length=11)
df['ema75'] = ta.ema(df['Close_adj'], length=75)
df['atr'] = ta.atr(df['High_adj'], df['Low_adj'], df['Close_adj'], length=14)

# Filtros de Tendencia (OR en Pine Script)
ema_filt_short = (df['Close_adj'] < df['ema11']) | (df['Close_adj'] < df['ema75'])
ema_filt_long  = (df['Close_adj'] > df['ema11']) | (df['Close_adj'] > df['ema75'])

# DMI / ADX
adx_df = ta.adx(df['High_adj'], df['Low_adj'], df['Close_adj'], length=14)
df['adx'] = adx_df['ADX_14']

# Calidad de Cierre (QC)
df['range'] = df['High_adj'] - df['Low_adj']
df['ubicacion_cierre'] = np.where(df['range'] == 0, 0.5, (df['Close_adj'] - df['Low_adj']) / df['range'])

# --- 3. LÓGICA DE MICRO-PAUSA Y ESTRUCTURA ---
df['v1_high'] = df['High_adj'].shift(1)
df['v1_low'] = df['Low_adj'].shift(1)
df['v1_close'] = df['Close_adj'].shift(1)
df['v1_open'] = df['Open_adj'].shift(1)

df['v2_high'] = df['High_adj'].shift(2)
df['v2_low'] = df['Low_adj'].shift(2)

# Condiciones de pausa
long_pause = (df['v1_high'] < df['v2_high']) & (df['v1_low'] < df['v2_low'])
long_v2_ok = df['v1_close'] > df['v1_open']
long_trigger = df['Close_adj'] > df['v1_high']

short_pause = (df['v1_low'] > df['v2_low']) & (df['v1_high'] > df['v2_high'])
short_v2_ok = df['v1_close'] < df['v1_open']
short_trigger = df['Close_adj'] < df['v1_low']

# Filtros Globales
momentum_ok = (df['adx'] >= 23) & (df['atr'] >= 2.5) & (df['atr'] <= 15.0)
qc_threshold = 0.8

# --- 4. SEÑALES FINALES ---
df['sig_long'] = long_pause & long_v2_ok & long_trigger & momentum_ok & ema_filt_long & (df['ubicacion_cierre'] >= qc_threshold) 
df['sig_short'] = short_pause & short_v2_ok & short_trigger & momentum_ok & ema_filt_short & (df['ubicacion_cierre'] <= (1-qc_threshold)) 

# --- 5. CÁLCULO DE NIVELES DESACOPLADOS (La Clave de la Sincronía) ---
# SL Estructural
df['sl_level'] = np.where(df['sig_long'], df['v2_low'],
                 np.where(df['sig_short'], df['v2_high'], np.nan))

# Riesgo y TP (Calculados sobre el cierre de la señal como en Pine)
df['risk_pts_signal'] = abs(df['Close_adj'] - df['sl_level'])

df['tp_level'] = np.where(df['sig_long'], df['Close_adj'] + df['risk_pts_signal'],
                 np.where(df['sig_short'], df['Close_adj'] - df['risk_pts_signal'], np.nan))

# Redondeo oficial al tick del Oro (0.1)
df['sl_level'] = (df['sl_level'] / 0.1).round() * 0.1
df['tp_level'] = (df['tp_level'] / 0.1).round() * 0.1

# --- 6. FUNCIONES DE EJECUCIÓN ---
def analyze_specific_day(engine, target_date_str):
    print(f"\n--- 🔍 AUDITORÍA DETALLADA: {target_date_str} ---")
    start = pd.to_datetime(target_date_str + " 00:00:00")
    end = pd.to_datetime(target_date_str + " 23:59:59")
    
    # Diagnóstico rápido antes de correr
    check = df[(df['Timestamp_NY'] >= f"{target_date_str} 09:41:00") & (df['Timestamp_NY'] <= f"{target_date_str} 09:43:00")]
    if not check.empty:
        print("Validando niveles de señal técnica:")
        print(check[['Timestamp_NY', 'Close_adj', 'sl_level', 'tp_level', 'sig_short']])
        
    return engine.run(start_date=start, end_date=end, verbose=True)

# --- 7. LANZAMIENTO ---
config = StrategyConfig(
    asset_name="GC",
    risk_usd=2000,
    reward_ratio=1.0,
    be_trigger_r=10.0, 
    be_offset_ticks=0,
    trading_windows=[("09:30", "10:30")],
    max_trades_per_day=-1
)

engine = QuantEngineV2(df, config)
#res = analyze_specific_day(engine, "2025-12-08")
trades = engine.run(start_date='2025-01-01', end_date='2025-12-31')


if trades is not None and not trades.empty:
    reporter = QuantReporter(trades)
    reporter.print_report()


       INFORME DE RENDIMIENTO QUANT
Total Trades        :      85.00
Win Rate (%)        :      72.94
Expectancy (R)      :       0.39
Profit Factor (R)   :       2.37
Max Drawdown (R)    :      -2.28
Recovery Factor     :      14.55
Sharpe Ratio (R)    :       0.44
Total PnL (R)       :      33.16
Total PnL (USD)     :   58574.00
Avg Trade (USD)     :     689.11


In [47]:
if not trades.empty:
    # Añadimos 'exit_time' para ver si el trade previo bloqueó al de las 09:48
    detalle_trades = trades[['id', 'entry_time', 'exit_time', 'type', 'entry', 'exit', 'pnl_usd','pnl_r', 'reason']]
    
    print(f"\nLista detallada de trades en Diciembre (Total: {len(trades)}):")
    # Formateamos para que las horas sean legibles
    print(detalle_trades.to_string(index=False))


Lista detallada de trades en Diciembre (Total: 85):
 id          entry_time           exit_time  type  entry   exit  pnl_usd     pnl_r reason
  1 2025-02-14 10:27:00 2025-02-14 10:32:00 Short 2723.5 2718.6   1429.8  0.934510     TP
  2 2025-03-21 09:48:00 2025-03-21 09:57:00 Short 2825.9 2820.9   1459.8  0.935769     TP
  3 2025-03-21 10:18:00 2025-03-21 10:22:00  Long 2820.5 2824.8   1666.4  0.925778     TP
  4 2025-04-03 10:13:00 2025-04-03 10:25:00  Long 2978.5 2985.8   1433.2  0.955467     TP
  5 2025-04-04 10:13:00 2025-04-04 10:15:00 Short 2905.9 2900.4   1609.8  0.941404     TP
  6 2025-04-04 10:22:00 2025-04-04 10:24:00  Long 2911.7 2916.5   1866.4  0.933200     TP
  7 2025-04-04 10:25:00 2025-04-04 10:28:00 Short 2910.8 2906.8   1546.4  0.920476     TP
  8 2025-04-08 09:49:00 2025-04-08 09:51:00 Short 2859.6 2856.8   1599.6  0.888667     TP
  9 2025-04-09 09:30:00 2025-04-09 09:33:00  Long 2919.5 2923.6   1586.4  0.922326     TP
 10 2025-04-09 09:33:00 2025-04-09 09:37:00  Lo

In [40]:
# 1. Parámetros del trade según tu lista
# Trade #4: Entrada 09:48, Salida 09:49 (Python)
trade_id = 4
target_day = "2025-12-08"
entry_time = "09:48:00"

# 2. Extraemos el trade y las velas relevantes
trade_data = df[(df['Timestamp_NY'] >= f"{target_day} 09:48:00") & 
                (df['Timestamp_NY'] <= f"{target_day} 09:51:00")].copy()

# 3. Recuperamos los niveles que calculó la señal en la vela de las 09:48
levels = trade_data[trade_data['Timestamp_NY'] == f"{target_day} {entry_time}"].iloc[0]
sl_level = levels['sl_level']
tp_level = levels['tp_level']
entry_p = levels['Close_adj'] # Asumiendo entrada al cierre + slippage 0

print(f"--- 🔍 AUDITORÍA MICRO TRADER #{trade_id} ---")
print(f"Entrada: {entry_p} | SL: {sl_level} | TP: {tp_level}\n")

cols = ['Timestamp_NY', 'Open_adj', 'High_adj', 'Low_adj', 'Close_adj']
print(trade_data[cols].to_string(index=False))

# 4. Verificación lógica manual
for i, row in trade_data.iterrows():
    print(f"\nRevisando vela {row['Timestamp_NY'].time()}:")
    if row['High_adj'] >= sl_level:
        print(f"  🔴 [SL TOUCHED] High {row['High_adj']} >= SL {sl_level}")
    if row['Low_adj'] <= tp_level:
        print(f"  🟢 [TP TOUCHED] Low {row['Low_adj']} <= TP {tp_level}")

--- 🔍 AUDITORÍA MICRO TRADER #4 ---
Entrada: 4189.7 | SL: 4192.2 | TP: 4187.2

       Timestamp_NY  Open_adj  High_adj  Low_adj  Close_adj
2025-12-08 09:48:00    4190.9    4192.0   4189.2     4189.7
2025-12-08 09:49:00    4189.4    4192.5   4189.0     4192.0
2025-12-08 09:50:00    4192.3    4194.4   4191.6     4193.4
2025-12-08 09:51:00    4193.7    4194.2   4191.1     4192.3

Revisando vela 09:48:00:

Revisando vela 09:49:00:
  🔴 [SL TOUCHED] High 4192.5 >= SL 4192.2

Revisando vela 09:50:00:
  🔴 [SL TOUCHED] High 4194.4 >= SL 4192.2

Revisando vela 09:51:00:
  🔴 [SL TOUCHED] High 4194.2 >= SL 4192.2


In [25]:
# 1. Identificar nombres de columnas reales (ignorando mayúsculas/minúsculas)
all_cols = df.columns.tolist()
def find_col(name):
    for c in all_cols:
        if c.lower() == name.lower(): return c
    return None

# 2. Construir la lista de columnas existentes
cols_to_show = [find_col('Timestamp_NY'), find_col('sig_short'), find_col('sig_long'), 
                find_col('close'), find_col('high'), find_col('low')]
# Limpiar Nones en caso de que alguna no exista
cols_to_show = [c for c in cols_to_show if c is not None]

# 3. Añadir indicadores técnicos si existen
for tech in ['ema1', 'ema2', 'adx', 'atr']:
    col = find_col(tech)
    if col: cols_to_show.append(col)

# 4. Filtrar y mostrar
debug_df = df[(df[find_col('Timestamp_NY')] >= '2025-12-29 09:45:00') & 
              (df[find_col('Timestamp_NY')] <= '2025-12-29 09:55:00')]

print(f"Columnas detectadas: {cols_to_show}")
print("\nDetalle de señales y filtros (09:45 - 09:55):")
print(debug_df[cols_to_show].to_string(index=False))

Columnas detectadas: ['Timestamp_NY', 'sig_short', 'sig_long', 'Close', 'High', 'Low', 'adx', 'atr']

Detalle de señales y filtros (09:45 - 09:55):
       Timestamp_NY  sig_short  sig_long  Close   High    Low       adx      atr
2025-12-29 09:45:00      False     False 4398.5 4399.0 4391.8 34.410702 7.411099
2025-12-29 09:46:00      False     False 4402.1 4402.8 4397.5 32.226514 7.260306
2025-12-29 09:47:00      False     False 4401.6 4403.9 4401.0 30.049881 6.948856
2025-12-29 09:48:00      False     False 4395.5 4403.3 4395.5 28.711519 7.009652
2025-12-29 09:49:00      False     False 4395.6 4397.5 4392.6 27.800421 6.858962
2025-12-29 09:50:00      False      True 4397.6 4398.0 4392.8 26.874659 6.740465
2025-12-29 09:51:00      False     False 4389.2 4396.9 4389.2 26.437577 6.859003
2025-12-29 09:52:00      False     False 4389.3 4394.8 4388.3 26.135634 6.833360
2025-12-29 09:53:00      False     False 4392.9 4393.8 4388.1 25.879563 6.752406
2025-12-29 09:54:00      False     False 4

In [28]:
# 1. Localizar la fila
row_48 = df[df['Timestamp_NY'] == '2025-12-29 09:48:00'].iloc[0]

# 2. Identificar columnas de EMA automáticamente
ema_cols = [c for c in df.columns if 'ema' in c.lower()]
price_close = row_48['Close']

print(f"--- Análisis de Filtros 09:48 ---")
print(f"Precio de Cierre: {price_close}")

# 3. Verificar EMAs
if not ema_cols:
    print("❌ No se encontraron columnas con el nombre 'ema'. Revisa df.columns")
else:
    for col in ema_cols:
        val = row_48[col]
        estado = "BAJISTA (OK para Short)" if price_close < val else "ALCISTA (Bloquea Short)"
        print(f"Filtro {col}: {val:.2f} -> {estado}")

# 4. Otros filtros (asegurando nombres correctos)
for tech in ['adx', 'atr']:
    col_name = next((c for c in df.columns if tech in c.lower()), None)
    if col_name:
        print(f"Filtro {tech.upper()}: {row_48[col_name]:.2f}")

--- Análisis de Filtros 09:48 ---
Precio de Cierre: 4395.5
Filtro ema11: 4356.46 -> ALCISTA (Bloquea Short)
Filtro ema75: 4380.29 -> ALCISTA (Bloquea Short)
Filtro ADX: 28.71
Filtro ATR: 7.01


In [4]:
# --- 1. CONFIGURACIÓN DEL RANGO ---
start_month = "2025-12-01 00:00:00"
end_month   = "2025-12-31 23:59:59"

print(f"🚀 Iniciando Backtest Mensual: {start_month} al {end_month}")

# --- 2. EJECUCIÓN DEL MOTOR ---
# Usamos verbose=False para que no imprima cada vela, solo el resumen final
# 'res_mensual' contendrá la lista de todos los trades del mes
res_mensual = engine.run(
    start_date=pd.to_datetime(start_month), 
    end_date=pd.to_datetime(end_month), 
    verbose=False 
)

# --- 3. REPORTE CONSOLIDADO ---
if res_mensual is not None and not res_mensual.empty:
    print("\n" + "="*40)
    print("       RESULTADOS DICIEMBRE 2025")
    print("="*40)
    
    reporter = QuantReporter(res_mensual)
    reporter.print_report()
    
    # Opcional: Ver los mejores y peores días
    res_mensual['Date'] = res_mensual['Exit_Time'].dt.date
    pnl_diario = res_mensual.groupby('Date')['PnL_USD'].sum()
    print("\n--- 📅 Resumen por Día (USD) ---")
    print(pnl_diario)
else:
    print("\n[!] No se ejecutaron trades en el periodo seleccionado.")

🚀 Iniciando Backtest Mensual: 2025-12-01 00:00:00 al 2025-12-31 23:59:59

       RESULTADOS DICIEMBRE 2025

       INFORME DE RENDIMIENTO QUANT
Total Trades        :      18.00
Win Rate (%)        :       0.00
Strike Rate (%)     :       0.00
Expectancy (R)      :      -0.00
Profit Factor       :       0.93
Max Drawdown (R)    :      -0.29
Total PnL (R)       :      -0.05
Total PnL (USD)     :    -106.80
Avg Trade (USD)     :      -5.93


KeyError: 'Exit_Time'

In [34]:
# 1. Primero asegúrate de que la función esté definida
def audit_signals(df_to_audit, config_to_use):
    print("\n" + "="*50)
    print("🔍 AUDITORÍA DE SEÑALES Y CONFIGURACIÓN")
    print("="*50)
    
    # Conteo
    longs = df_to_audit['sig_long'].sum() if 'sig_long' in df_to_audit.columns else 0
    shorts = df_to_audit['sig_short'].sum() if 'sig_short' in df_to_audit.columns else 0
    
    print(f"Señales en Columnas: Long: {longs} | Short: {shorts}")

    # Verificar si las señales ocurren en las ventanas de tiempo
    df_signals = df_to_audit[(df_to_audit['sig_long'] == True) | (df_to_audit['sig_short'] == True)].copy()
    
    if df_signals.empty:
        print("❌ RESULTADO: No hay ninguna señal (True) en las columnas sig_long/sig_short.")
    else:
        # Check de ventanas
        from datetime import time
        def check_time(dt):
            for start_str, end_str in config_to_use.trading_windows:
                if time.fromisoformat(start_str) <= dt.time() <= time.fromisoformat(end_str):
                    return True
            return False
        
        df_signals['in_window'] = df_signals['Timestamp_NY'].apply(check_time)
        print(f"Señales Totales: {len(df_signals)}")
        print(f"Señales en Ventana Permitida: {df_signals['in_window'].sum()}")
        
        if df_signals['in_window'].sum() > 0:
            print("\nMuestra de señales válidas:")
            print(df_signals[df_signals['in_window'] == True][['Timestamp_NY', 'Close_adj', 'sl_level']].tail(5))
    print("="*50 + "\n")

# 2. EJECUCIÓN (Ajusta los nombres de tus variables aquí)
# Si tu dataframe se llama 'df_operativo' y tu config 'mi_estrategia', cámbialos abajo:
try:
    audit_signals(df, config)  # <--- Cambia 'df' y 'config' si tienen otros nombres
except NameError as e:
    print(f"❌ Error: {e}. Revisa cómo se llaman tus variables de DataFrame y Configuración.")


🔍 AUDITORÍA DE SEÑALES Y CONFIGURACIÓN
Señales en Columnas: Long: 443 | Short: 450
Señales Totales: 893
Señales en Ventana Permitida: 137

Muestra de señales válidas:
               Timestamp_NY  Close_adj  sl_level
5473565 2026-01-27 10:07:00     5019.6    5025.9
5474912 2026-01-28 09:34:00     5295.3    5284.1
5474955 2026-01-28 10:17:00     5279.6    5298.6
5480437 2026-02-03 09:39:00     4942.0    4927.7
5481859 2026-02-04 10:22:00     4976.6    5006.1



In [10]:
import pandas as pd

# --- 1. CONFIGURACIÓN DEL FILTRO ---
fecha_target = "2025-12-08"
hora_inicio = "09:30:00"
hora_fin = "10:30:00"

# --- 2. FILTRADO DINÁMICO ---
# Aseguramos que el índice o columna sea datetime
df_view = df.copy()
mask = (df_view['Timestamp_NY'] >= f"{fecha_target} {hora_inicio}") & \
       (df_view['Timestamp_NY'] <= f"{fecha_target} {hora_fin}")

cols_interes = ['Timestamp_NY', 'Open_adj', 'High_adj', 'Low_adj', 'Close_adj', 'sig_long', 'sig_short', 'sl_level']
df_filtrado = df_view.loc[mask, cols_interes].reset_index(drop=True)

# --- 3. VISUALIZACIÓN ESTILIZADA ---
def color_signals(val):
    if val is True: return 'background-color: #2e7d32; color: white' # Verde para señales
    return ''

print(f"📊 Inspección de Microestructura: {fecha_target} ({hora_inicio} - {hora_fin})")

# Mostramos el DF con formato para detectar el Low de las 09:45
df_filtrado.style.format({
    'Open_adj': '{:.2f}', 'High_adj': '{:.2f}', 
    'Low_adj': '{:.2f}', 'Close_adj': '{:.2f}',
    'sl_level': '{:.2f}'
}).applymap(color_signals, subset=['sig_long', 'sig_short'])

📊 Inspección de Microestructura: 2025-12-08 (09:30:00 - 10:30:00)


/tmp/ipykernel_162522/3371399346.py:29: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  }).applymap(color_signals, subset=['sig_long', 'sig_short'])


,Timestamp_NY,Open_adj,High_adj,Low_adj,Close_adj,sig_long,sig_short,sl_level
0,2025-12-08 09:30:00,4199.30,4200.20,4198.00,4199.30,False,False,nan
1,2025-12-08 09:31:00,4199.40,4200.30,4197.70,4200.00,False,False,nan
2,2025-12-08 09:32:00,4199.70,4201.50,4199.40,4200.00,False,False,nan
3,2025-12-08 09:33:00,4200.00,4200.20,4195.30,4196.00,False,False,nan
4,2025-12-08 09:34:00,4195.70,4197.70,4194.30,4196.90,False,False,nan
5,2025-12-08 09:35:00,4196.70,4198.40,4195.40,4197.40,False,False,nan
6,2025-12-08 09:36:00,4197.30,4197.70,4193.50,4194.00,False,False,nan
7,2025-12-08 09:37:00,4194.10,4194.90,4192.40,4193.60,False,False,nan
8,2025-12-08 09:38:00,4193.40,4195.10,4192.50,4195.10,False,False,nan
9,2025-12-08 09:39:00,4195.00,4195.50,4193.40,4195.40,False,False,nan


In [12]:
import pandas as pd

# 1. Filtro temporal estricto
target_day = "2025-12-08"
start_t = f"{target_day} 09:40:00"
end_t = f"{target_day} 09:55:00"

# 2. Selección de columnas de precio y señales
# Incluimos las señales crudas para ver por qué el trade entra a las 09:43
cols = ['Timestamp_NY', 'Open_adj', 'High_adj', 'Low_adj', 'Close_adj', 'sig_long', 'sig_short', 'sl_level']

df_inspect = df[(df['Timestamp_NY'] >= start_t) & (df['Timestamp_NY'] <= end_t)][cols].copy()

# 3. Formateo para lectura rápida
print(f"🔍 AUDITORÍA DE DATOS RAW - {target_day}")
display(df_inspect.style.format({
    'Open_adj': '{:.2f}', 'High_adj': '{:.2f}', 
    'Low_adj': '{:.2f}', 'Close_adj': '{:.2f}', 
    'sl_level': '{:.2f}'
}).highlight_between(left=True, right=True, subset=['sig_short'], color='#ffcccc'))

🔍 AUDITORÍA DE DATOS RAW - 2025-12-08


,Timestamp_NY,Open_adj,High_adj,Low_adj,Close_adj,sig_long,sig_short,sl_level
5426964,2025-12-08 09:40:00,4195.70,4196.90,4194.00,4196.40,False,False,nan
5426965,2025-12-08 09:41:00,4196.50,4197.10,4194.70,4194.80,False,False,nan
5426966,2025-12-08 09:42:00,4194.80,4194.80,4190.40,4191.00,False,True,4196.90
5426967,2025-12-08 09:43:00,4190.90,4191.80,4189.90,4191.50,False,False,nan
5426968,2025-12-08 09:44:00,4191.50,4192.50,4188.60,4188.60,False,False,nan
5426969,2025-12-08 09:45:00,4188.80,4191.20,4185.00,4190.50,False,False,nan
5426970,2025-12-08 09:46:00,4190.00,4192.20,4187.90,4192.00,False,False,nan
5426971,2025-12-08 09:47:00,4191.60,4193.80,4190.60,4191.20,False,False,nan
5426972,2025-12-08 09:48:00,4190.90,4192.00,4189.20,4189.70,False,True,4192.20
5426973,2025-12-08 09:49:00,4189.40,4192.50,4189.00,4192.00,False,False,nan


In [13]:
import pandas as pd

# 1. Definimos el rango de interés (Vela del TP fallido y entrada)
target_day = "2025-12-08"
start_t = f"{target_day} 09:40:00"
end_t = f"{target_day} 09:50:00"

# 2. Selección de columnas: Precios Crudos vs Ajustados y Contrato
# Nota: Ajusta 'symbol' o 'contract_name' según como se llame en tu parquet
cols_raw = ['Timestamp_NY', 'Open', 'High', 'Low', 'Close', 'Open_adj', 'High_adj', 'Low_adj', 'Close_adj']

# Buscamos si existe una columna de contrato/símbolo
potential_contract_cols = ['symbol', 'contract', 'contract_name', 'ticker']
available_cols = [c for c in potential_contract_cols if c in df.columns]
final_cols = cols_raw + available_cols

df_raw_inspect = df[(df['Timestamp_NY'] >= start_t) & (df['Timestamp_NY'] <= end_t)][final_cols].copy()

# 3. Visualización
print(f"📋 COMPARATIVA DE PRECIOS CRUDOS VS AJUSTADOS - {target_day}")
if not available_cols:
    print("⚠️ Nota: No se encontró una columna explícita de 'contrato'. Se asume GC continuo.")

display(df_raw_inspect.style.format({
    'Open': '{:.2f}', 'High': '{:.2f}', 'Low': '{:.2f}', 'Close': '{:.2f}',
    'Open_adj': '{:.2f}', 'High_adj': '{:.2f}', 'Low_adj': '{:.2f}', 'Close_adj': '{:.2f}'
}).background_gradient(subset=['Low', 'Low_adj'], cmap='YlOrRd'))

📋 COMPARATIVA DE PRECIOS CRUDOS VS AJUSTADOS - 2025-12-08
⚠️ Nota: No se encontró una columna explícita de 'contrato'. Se asume GC continuo.


,Timestamp_NY,Open,High,Low,Close,Open_adj,High_adj,Low_adj,Close_adj
5426964,2025-12-08 09:40:00,4234.50,4235.70,4232.80,4235.20,4195.70,4196.90,4194.00,4196.40
5426965,2025-12-08 09:41:00,4235.30,4235.90,4233.50,4233.60,4196.50,4197.10,4194.70,4194.80
5426966,2025-12-08 09:42:00,4233.60,4233.60,4229.20,4229.80,4194.80,4194.80,4190.40,4191.00
5426967,2025-12-08 09:43:00,4229.70,4230.60,4228.70,4230.30,4190.90,4191.80,4189.90,4191.50
5426968,2025-12-08 09:44:00,4230.30,4231.30,4227.40,4227.40,4191.50,4192.50,4188.60,4188.60
5426969,2025-12-08 09:45:00,4227.60,4230.00,4223.80,4229.30,4188.80,4191.20,4185.00,4190.50
5426970,2025-12-08 09:46:00,4228.80,4231.00,4226.70,4230.80,4190.00,4192.20,4187.90,4192.00
5426971,2025-12-08 09:47:00,4230.40,4232.60,4229.40,4230.00,4191.60,4193.80,4190.60,4191.20
5426972,2025-12-08 09:48:00,4229.70,4230.80,4228.00,4228.50,4190.90,4192.00,4189.20,4189.70
5426973,2025-12-08 09:49:00,4228.20,4231.30,4227.80,4230.80,4189.40,4192.50,4189.00,4192.00


In [14]:
import pandas as pd

# 1. Definición del rango de auditoría
fecha_target = "2025-12-08"
inicio = f"{fecha_target} 09:30:00"
fin = f"{fecha_target} 10:30:00"

# 2. Selección de columnas críticas para el diagnóstico
# Incluimos Ticker (Nombre del contrato) y Adj_Factor (Monto del ajuste)
cols_inspeccion = [
    'Timestamp_NY', 'Ticker', 'Adj_Factor',
    'Open', 'High', 'Low', 'Close', 
    'Open_adj', 'High_adj', 'Low_adj', 'Close_adj'
]

# 3. Filtrado del DataFrame
df_audit = df[(df['Timestamp_NY'] >= inicio) & (df['Timestamp_NY'] <= fin)][cols_inspeccion].copy()

# 4. Visualización formateada
print(f"🔍 AUDITORÍA DE MICROESTRUCTURA - {fecha_target}")
print(f"Contrato detectado en el periodo: {df_audit['Ticker'].unique()}")

# Estilizamos para encontrar el Low de las 09:45
display(df_audit.style.format({
    'Adj_Factor': '{:.1f}',
    'Open': '{:.2f}', 'High': '{:.2f}', 'Low': '{:.2f}', 'Close': '{:.2f}',
    'Open_adj': '{:.2f}', 'High_adj': '{:.2f}', 'Low_adj': '{:.2f}', 'Close_adj': '{:.2f}'
}).background_gradient(subset=['Low', 'Low_adj'], cmap='YlGnBu'))

🔍 AUDITORÍA DE MICROESTRUCTURA - 2025-12-08
Contrato detectado en el periodo: ['GCG6']


,Timestamp_NY,Ticker,Adj_Factor,Open,High,Low,Close,Open_adj,High_adj,Low_adj,Close_adj
5426954,2025-12-08 09:30:00,GCG6,-38.8,4238.10,4239.00,4236.80,4238.10,4199.30,4200.20,4198.00,4199.30
5426955,2025-12-08 09:31:00,GCG6,-38.8,4238.20,4239.10,4236.50,4238.80,4199.40,4200.30,4197.70,4200.00
5426956,2025-12-08 09:32:00,GCG6,-38.8,4238.50,4240.30,4238.20,4238.80,4199.70,4201.50,4199.40,4200.00
5426957,2025-12-08 09:33:00,GCG6,-38.8,4238.80,4239.00,4234.10,4234.80,4200.00,4200.20,4195.30,4196.00
5426958,2025-12-08 09:34:00,GCG6,-38.8,4234.50,4236.50,4233.10,4235.70,4195.70,4197.70,4194.30,4196.90
5426959,2025-12-08 09:35:00,GCG6,-38.8,4235.50,4237.20,4234.20,4236.20,4196.70,4198.40,4195.40,4197.40
5426960,2025-12-08 09:36:00,GCG6,-38.8,4236.10,4236.50,4232.30,4232.80,4197.30,4197.70,4193.50,4194.00
5426961,2025-12-08 09:37:00,GCG6,-38.8,4232.90,4233.70,4231.20,4232.40,4194.10,4194.90,4192.40,4193.60
5426962,2025-12-08 09:38:00,GCG6,-38.8,4232.20,4233.90,4231.30,4233.90,4193.40,4195.10,4192.50,4195.10
5426963,2025-12-08 09:39:00,GCG6,-38.8,4233.80,4234.30,4232.20,4234.20,4195.00,4195.50,4193.40,4195.40


In [5]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import pandas as pd
import numpy as np
import pandas_ta as ta
from quant_backtester_core import QuantEngineV2, StrategyConfig
from quant_reporting import QuantReporter

# --- 1. CARGA DE DATOS ---
FILE_PATH_PARQUET = '/home/quant/data/processed/gc_1m_raw_continuous.parquet'
df = pd.read_parquet(FILE_PATH_PARQUET)

# Asegurar formato de tiempo
df['Timestamp_NY'] = pd.to_datetime(df['Timestamp_NY'])
df = df.sort_values('Timestamp_NY').reset_index(drop=True)

# --- 2. CÁLCULO DE INDICADORES (Lógica Pine Script) ---
# Filtros de tendencia y momentum
df['ema11'] = ta.ema(df['Close_adj'], length=11)
df['ema75'] = ta.ema(df['Close_adj'], length=75)
df['atr'] = ta.atr(df['High_adj'], df['Low_adj'], df['Close_adj'], length=14)

# DMI / ADX
adx_df = ta.adx(df['High_adj'], df['Low_adj'], df['Close_adj'], length=14)
df['adx'] = adx_df['ADX_14']

# Calidad de Cierre (QC)
df['range'] = df['High_adj'] - df['Low_adj']
df['ubicacion_cierre'] = np.where(df['range'] == 0, 0.5, (df['Close_adj'] - df['Low_adj']) / df['range'])

# --- 3. LÓGICA DE MICRO-PAUSA ---
# Capturamos valores previos con shift para v1 y v2
df['v1_high'] = df['High_adj'].shift(1)
df['v1_low'] = df['Low_adj'].shift(1)
df['v1_close'] = df['Close_adj'].shift(1)
df['v1_open'] = df['Open_adj'].shift(1)

df['v2_high'] = df['High_adj'].shift(2)
df['v2_low'] = df['Low_adj'].shift(2)

# Condiciones de pausa estructural
long_pause = (df['v1_high'] < df['v2_high']) & (df['v1_low'] < df['v2_low'])
long_v2_ok = df['v1_close'] > df['v1_open']
long_trigger = df['Close_adj'] > df['v1_high']

short_pause = (df['v1_low'] > df['v2_low']) & (df['v1_high'] > df['v2_high'])
short_v2_ok = df['v1_close'] < df['v1_open']
short_trigger = df['Close_adj'] < df['v1_low']

# Filtros Globales (Momentum y Calidad)
momentum_ok = (df['adx'] >= 23) & (df['atr'] >= 2.5) & (df['atr'] <= 15.0)
qc_threshold = 0.8

# Señales Finales
# Se añade filtro de EMA11 para coincidir con el script: "close > ema11_val"
df['sig_long'] = long_pause & long_v2_ok & long_trigger & momentum_ok & (df['ubicacion_cierre'] >= qc_threshold) & (df['Close_adj'] > df['ema11'])
df['sig_short'] = short_pause & short_v2_ok & short_trigger & momentum_ok & (df['ubicacion_cierre'] <= (1-qc_threshold)) & (df['Close_adj'] < df['ema11'])
# ========================================================
# INSERTA AQUÍ LA LÓGICA DE NIVELES (ANTES DEL MOTOR)
# ========================================================

# 1. SL Estructural
df['sl_level'] = np.where(df['sig_long'], df['v2_low'],
                 np.where(df['sig_short'], df['v2_high'], np.nan))

# 2. Riesgo desde el CIERRE (Sincronía con Pine Script)
df['risk_pts_signal'] = abs(df['Close_adj'] - df['sl_level'])

# 3. TP desde el CIERRE (RR 1.0)
df['tp_level'] = np.where(df['sig_long'], df['Close_adj'] + df['risk_pts_signal'],
                 np.where(df['sig_short'], df['Close_adj'] - df['risk_pts_signal'], np.nan))

# 4. REDONDEO AL TICK (GC = 0.1)
# Esto garantiza que el TP sea 4185.1 en lugar de 4184.9
df['sl_level'] = (df['sl_level'] / 0.1).round() * 0.1
df['tp_level'] = (df['tp_level'] / 0.1).round() * 0.1

# ========================================================

# ========================================================
# 🔍 CONTROL DE CALIDAD PRE-MOTOR
# ========================================================
print("Verificando Niveles del Trade #1...")
check_trade = df[(df['Timestamp_NY'] >= '2025-12-08 09:41:00') & (df['Timestamp_NY'] <= '2025-12-08 09:43:00')]
print(check_trade[['Timestamp_NY', 'Close_adj', 'sl_level', 'tp_level', 'sig_short']])
# ========================================================


def analyze_specific_day(engine, target_date_str):
    print(f"\n--- 🔍 AUDITORÍA DETALLADA: {target_date_str} ---")
    start = pd.to_datetime(target_date_str + " 00:00:00")
    end = pd.to_datetime(target_date_str + " 23:59:59")
    return engine.run(start_date=start, end_date=end, verbose=True)
# --- 4. EJECUCIÓN DEL BACKTEST ---
config = StrategyConfig(
    asset_name="GC",
    risk_usd=2000,
    reward_ratio=1.0,
    be_trigger_r=10.0, 
    be_offset_ticks=0,
    trading_windows=[("09:30", "10:30")],
    max_trades_per_day=-1,  # Sin límite
    execution_mode="Optimista"
)

engine = QuantEngineV2(df, config)

# Ejecutamos el análisis del día 8 y capturamos en 'res'
#res = analyze_specific_day(engine, "2025-12-08")
# Ejecutamos para el mes de Diciembre 2025
trades = engine.run(start_date='2025-12-01', end_date='2025-12-31')



# --- 5. REPORTING Y CONTRASTE ---
if res is not None and not res.empty:
    # IMPORTANTE: Usamos 'res', no 'trades'
    reporter = QuantReporter(res)
    reporter.print_report()
else:
    print(f"\n[!] El motor terminó el día sin ejecutar trades.")




# Descomentar para visualizar gráficos
# reporter.plot_equity_curve()

Verificando Niveles del Trade #1...
               Timestamp_NY  Close_adj  sl_level  tp_level  sig_short
5426965 2025-12-08 09:41:00     4194.8       NaN       NaN      False
5426966 2025-12-08 09:42:00     4191.0    4196.9    4185.1       True
5426967 2025-12-08 09:43:00     4191.5       NaN       NaN      False

       INFORME DE RENDIMIENTO QUANT
Total Trades        :       3.00
Win Rate (%)        :       0.00
Strike Rate (%)     :       0.00
Expectancy (R)      :      -0.07
Profit Factor       :       0.23
Max Drawdown (R)    :      -0.29
Total PnL (R)       :      -0.22
Total PnL (USD)     :    -445.80
Avg Trade (USD)     :    -148.60


In [12]:
# Buscamos si hay señal a las 09:30 o 09:29 el día 29
print(df[(df['Timestamp_NY'] >= '2025-12-29 09:29:00') & (df['Timestamp_NY'] <= '2025-12-29 09:31:00')][['Timestamp_NY', 'sig_short', 'sig_long']])

               Timestamp_NY  sig_short  sig_long
5446077 2025-12-29 09:29:00      False     False
5446078 2025-12-29 09:30:00       True     False
5446079 2025-12-29 09:31:00      False     False
